# Linear Neural Networks for Regression
Prima di implementare delle reti neurali profonde è importante capire come funzionano quelle semplici dove gli output sono direttamente collegati agli input.

## Linear Regression
I problemi di regression sono quelli dove vogliamo predirre un valore numerico, vedremo come esempio quello di fare una stima dei prezzi delle case in una determinata zona avendo come base alcuni valori.
Per far funzionare la rete avremo bisogno dei dati, di solito organizzati in colonne, fissiamo qualche termine:
- Example: Una riga della tabella, indica in questo caso una casa
- Feature: Una colonna della tabella sono i valori ad esempio gli anni della casa, la zona, i $m^2$ ecc...
- Target / Label: Il valore che vogliamo stimare.

In [ ]:
%matplotlib inline
import math
import time
import numpy as np
import torch

## Basics
Nel nostro esempio vogliamo calcolare il prezzo di una casa basandoci sull'area e l'etá. Nella Linear Regression inoltre assumiamo che il target sia una funzione lineare delle features ovvero una somma pesata: $$\text{price}=w_{\text{area}}*\text{area} + w_{\text{age}} * \text{age}+b$$
Qui chiamiamo le $w$ pesi e $b$ bias o offset, i pesi ci indicano l'influenza di ogni feature nella somma mentre il bias fa una stima del prezzo quando tutte le features stanno a 0.
Il nostro obiettivo é quindi, dato un dataset, trovare i migliori pesi e bias per fare in modo che il nostro modello preveda correttamente i prezzi.

Potremmo dover lavorare con dataset multidimensionali, indichiamo quindi con $d$ il numero delle features e con $\hat{y}$ la nostra predizione: $$\hat{y}=w_1x_1+...+w_dx_d+b$$
Raccogliendo tutte le features in un vettore $\pmb{x}\in\mathbb{R}_d$ e tutti i pesi in un vettore $\pmb{w}\in\mathbb{R}^d$ possiamo esprimere il nostro modello in modo compatto tramite il dot product tra $\pmb{w}$ e $\pmb{x}$: $$\hat{y}=w^Tx+b$$

Nella formula sopra peró il vettore $x$ corrisponde alle features di un solo example, spesso vogliamo fare riferimento a tutto il dataset di $n$ examples tramite la matrice $\pmb{X}\in\mathbb{R}^{n\times d}$. Questa matrice avrá quindi una riga per ogni example e una colonna per ogni feature, possiamo quindi esprimere la predizione: $$\hat{y}=Xw+b$$
Quindi date le features di un dataset di training $\pmb{x}$ e le corrispondenti label $\pmb{y}$ l'obiettivo della linear regression é quello di trovare i pesi $\pmb{w}$ e il bias $b$ in modo che date delle nuove features di un example ricavato dalla stessa distribuzione di $\pmb{X}$ il modello preveda i valori con un basso errore.

Prima di iniziare a cercare i migliori parametri peró ci servono due cose:
- Come misurare la qualitá di un modello
- Una procedura per aggiornare il modello per migliorare la sua qualitá

## Loss Function
La Loss Function serve a quantificare la distanza tra il valore reale e quello predetto dal modello, solitamente é un numero non negativo dove piú il numero é basso meglio é.
Per i problemi di regression la loss function piú comune é la squared error. Indichiamo, per un example $i$, la nostra predizione con $\hat{y}^{(i)}$ e il corrispondente label vero con $y^{(i)}$, la squared error é data da: $$l^{(i)}(w,b)=\frac{1}{2}\left(\hat{y}^{(i)}-y^{(i)}\right)^2$$

Per misurare la qualitá di un modello sull'intero dataset di $n$ examples facciamo semplicemente la media delle losses sul training set: $$L(w,b)=\frac{1}{n}\sum_{i=1}^n l^{(i)}(w,b)=\frac{1}{n}\sum_{i=1}^n \frac{1}{2} \left( w^Tx^{(i)} + b - y^{(i)} \right)^2$$

Quando alleniamo il modello cerchiamo i parametri $(w^*, b^*)$ che minimizzano la loss totale su tutti gli examples di training: $$w^*, b^*=\underset{w,b}{argmin} \ L(w,b)$$

## Analytic Solution
Nella linear regression, diversamente da altri modelli che vedremo, possiamo trovare i parametri ottimali applicando una semplice formula. Per prima cosa possiamo includere il bias $b$ nel parametro $w$ aggiungendo una colonna di tutti 1 alla design matrix.
A questo punto il nostro prediction problem diventa minimizzare $||y-Xw||^2$.
Per minimizzare la formula calcoliamo la derivata e poniamola uguale a 0:
$$\partial_w||y-Xw||^2=2X^T(Xw-y)=0$$

E quindi otteniamo $$X^Ty=X^TXw$$

Risolvendo per $w$ otteniamo la soluzione ottimale $$w^*=(X^TX)^{-1}X^Ty$$

Problemi semplici come questo ammettono queste soluzioni "analitiche" ma non siamo mai cosí fortunati in deep learning. Nella maggior parte dei casi otteniamo i valori ottimali provando, sbagliando e correggendo i pesi.

## Minibatch Stochastic Gradient Descent
Il metodo chiave per ottimizzare un modello é quello di iterare piú volte sul training set riducendo l'errore aggiornando i parametri nella direzione in cui diminuiscono la loss function. Questo algoritmo é chiamato _gradient descent_.

L'approccio piú naive consiste nel calcolare la derivata della loss function che consiste nella media delle loss calcolate in ogni example del dataset. Nella pratica peró questo approccio é molto lento, dobbiamo passare su tutto il dataset prima di fare un update.

Possiamo provare l'estremo opposto ovvero fare un update ad ogni example, l'algoritmo si chiama _stochastic gradient descent (SGD)_ e puó essere ottimale anche su dataset grandi. Ha vari problemi peró:
- I processori sono piú veloci a fare calcoli piuttosto che spostare dati, questo significa che ci vuole piú tempo a processare un singolo example piuttosto che piú example in un colpo solo.
- Alcuni layer funzionano bene soltanto se hanno accesso a piú examples contemporanetamente.

La soluzione é una via di mezzo, ovvero prendere una minibatch di examples, quanti dipende da vari fattori come la memoria o il numero di GPU o il numero di layer ecc...
Di solito peró un numero tra 32 e 256 va bene, preferibilmente una potenza di 2 abbastanza grande. Questo ci porta ad utilizzare il _minibatch stochastic gradient descent_

Nella sua forma base, in ogni iterazione $t$ prima calcoliamo una minibatch $\mathcal{B}_t$ di un numero fisso $|\mathcal{B}|$ di examples. Poi calcoliamo la derivata (gradient) della loss media sulla minibatch rispetto ai parametri del modello.
Infine moltiplichiamo il gradiente per un piccolo valore positivo predeterminato $\eta$ chiamato _learning rate_ e sottraiamo il risultato ottenuto dai parametri attuali.
Possiamo esprimere quello appena detto in questo modo:
$$(w,b)\leftarrow(w,b)-\frac{\eta}{|\mathcal{B}|}\sum_{i\in \mathcal{B}_t}\partial_{(w,b)}l^{(i)}(w,b)$$

La grandezza delle minibatch e il learning rate di solito sono definiti dall'utente e non vengono aggiornati nel training loop, questi vengono chiamati _hyperparameters_.
Dopo aver allenato il modello per un numero prestabilito di iterazioni, salviamo i parametri denotati con $\hat{w},\hat{b}$

## Predictions